# bar-figure

论文 / Paper: **Paper name**  
模板 / Template: `bar`  

修改 `PLOT_SPEC` 以调整内容，修改 `CHART_STYLE` 以调整外观。图例直接使用同一组柱形对象，因此颜色和 hatch 会保持同步。  
Modify `PLOT_SPEC` for content and `CHART_STYLE` for appearance. The legend uses the same bar patches, so color and hatch stay synchronized.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import tempfile

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    raise RuntimeError("Run this notebook with notebooks/ as the working directory")
WORKSPACE_ROOT = NOTEBOOK_DIR.parent
SOURCE_DIR = WORKSPACE_ROOT / "data" / "source"
FIGURE_DIR = WORKSPACE_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# figure-style：共享 base.mplstyle 的自包含副本。
# figure-style: self-contained copy of the shared base.mplstyle.
# 修改带注释的文本后，请重新运行本单元和绘图单元。
# Edit the annotated text, then rerun this cell and the plotting cells.
BASE_MPLSTYLE = r"""# 科研绘图基础样式 / Base style for scientific figures

# 画布 / Canvas
figure.figsize: 3.35, 2.70
figure.dpi: 100
figure.facecolor: white
figure.constrained_layout.use: True

# 字体排印 / Typography
font.family: sans-serif
font.sans-serif: Arial, Liberation Sans, DejaVu Sans
font.size: 8.5
axes.titlesize: 9.5
axes.labelsize: 10
axes.labelweight: bold

# 坐标轴 / Axes
axes.facecolor: white
axes.edgecolor: 8C8C8C
axes.linewidth: 0.65
axes.axisbelow: True
axes.grid: True
axes.spines.top: True
axes.spines.right: True

# 系列循环：颜色、marker 和线型列表必须等长 / Series cycle: lists must have equal length
# marker：v 下三角，o 圆，s 方块，^ 上三角，D 菱形，X 实心叉
# markers: v down-triangle, o circle, s square, ^ up-triangle, D diamond, X filled-X
axes.prop_cycle: cycler(color=['0C84C6', '41B7AC', 'FFA510', 'F74D4D', '2455A4', '002C53']) + cycler(marker=['v', 'o', 's', '^', 'D', 'X']) + cycler(linestyle=['-', '--', '-.', ':', '-', '--'])

# 刻度 / Ticks
xtick.labelsize: 8.5
ytick.labelsize: 8.5
xtick.major.size: 2
ytick.major.size: 2
xtick.major.width: 0.6
ytick.major.width: 0.6

# 线、marker 与色块 / Lines, markers, and patches
lines.linewidth: 1.3
lines.markersize: 4.2
patch.linewidth: 0.65
patch.edgecolor: white
patch.force_edgecolor: True

# 图例 / Legend
legend.fontsize: 8.5
legend.frameon: True
legend.fancybox: True
legend.framealpha: 0.96
legend.edgecolor: A6A6A6
legend.facecolor: white
legend.handlelength: 1.7
legend.handletextpad: 0.5
legend.borderpad: 0.45

# 网格 / Grid
grid.color: 9A9A9A
grid.linestyle: --
grid.linewidth: 0.5
grid.alpha: 0.5

# 导出 / Export
savefig.dpi: 300
savefig.facecolor: white
savefig.transparent: False
pdf.fonttype: 42
ps.fonttype: 42
svg.fonttype: none"""

def _rcparams_from_mplstyle_text(text):
    """解析原生 Matplotlib key，同时忽略空行和注释行。

    Parse native Matplotlib keys while ignoring blank/comment lines.
    """
    params = {}
    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if ":" not in raw_line:
            raise ValueError(f"Style line {line_number} must use 'key: value'")
        key, value = (part.strip() for part in raw_line.split(":", 1))
        if key not in matplotlib.rcParams.validate:
            raise KeyError(f"Unknown Matplotlib rcParam on style line {line_number}: {key}")
        params[key] = matplotlib.rcParams.validate[key](value)
    return params

BASE_STYLE_SOURCE = {'filename': 'base.mplstyle',
 'sha256': 'cc3e628ba81f2b83136580d00247d2c56f22547348a3e113e4e788cea75a8b93'}
PAPER_STYLE = _rcparams_from_mplstyle_text(BASE_MPLSTYLE)
_STYLE_CYCLE = PAPER_STYLE["axes.prop_cycle"].by_key()
STYLE_COLORS = _STYLE_CYCLE["color"]
STYLE_MARKERS = _STYLE_CYCLE.get("marker", ["o"] * len(STYLE_COLORS))
STYLE_LINESTYLES = _STYLE_CYCLE.get("linestyle", ["-"] * len(STYLE_COLORS))


In [ ]:
FIGURE_METADATA = {'schema_version': '1.0',
 'paper_name': 'Paper name',
 'figure_slug': 'bar-figure',
 'chart_type': 'bar',
 'claim': 'State the figure claim',
 'data_sources': [{'workspace_path': 'data/source/results.csv', 'sha256': 'computed-at-runtime'}],
 'transformations': ['No aggregation or normalization in the generated baseline; rows are plotted '
                     'directly'],
 'missing_values': {'policy': 'fail when a required plotting column contains missing values'},
 'uncertainty': {'type': 'none', 'reason': 'generated baseline does not infer uncertainty'},
 'axis_policy': {'x_scale': 'linear', 'y_scale': 'linear'},
 'dimensions_inches': [3.35, 2.3],
 'palette': {'provider': 'embedded-base-style',
             'source_kind': 'reusable-style',
             'id': 'base-style-cycle',
             'source_url': None,
             'colors': ['#0C84C6', '#41B7AC', '#FFA510', '#F74D4D', '#2455A4', '#002C53'],
             'base_style': {'filename': 'base.mplstyle',
                            'source_sha256': 'cc3e628ba81f2b83136580d00247d2c56f22547348a3e113e4e788cea75a8b93',
                            'embedded_sha256': 'computed-at-runtime'}},
 'outputs': ['bar-figure.pdf', 'bar-figure.svg', 'bar-figure.png']}


## 学生修改指南 / Student editing guide

`canvas` 控制背景、网格和边框；`marks` 控制颜色、hatch 和柱宽；`legend` 控制边框与位置；`axes` 控制标签、刻度、旋转、比例、范围和数值格式。除非在转换单元中明确聚合，否则每个类别和系列应只保留一个值。  

`canvas` controls background/grid/spines; `marks` controls color/hatch/bar width; `legend` controls frame and placement; `axes` controls labels, ticks, rotation, scale, limits, and number formatting. Preserve one value per category and series unless you explicitly aggregate in the transformation cell.

In [ ]:
SOURCE_FILE = SOURCE_DIR / "results.csv"
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(SOURCE_FILE)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

suffix = SOURCE_FILE.suffix.lower()
if suffix in {".csv", ".txt"}: source_data = pd.read_csv(SOURCE_FILE)
elif suffix == ".tsv": source_data = pd.read_csv(SOURCE_FILE, sep="\t")
elif suffix in {".xlsx", ".xls"}: source_data = pd.read_excel(SOURCE_FILE)
elif suffix == ".json": source_data = pd.read_json(SOURCE_FILE)
else: raise ValueError(f"Unsupported source format: {suffix}")
FIGURE_METADATA["data_sources"][0]["sha256"] = file_sha256(SOURCE_FILE)
source_data.head()


In [ ]:
# 修改这个映射后，请重新运行全部单元。
# Edit this mapping, then rerun all cells.
PLOT_SPEC = {'x_column': 'x',
 'y_column': 'value',
 'series_column': 'method',
 'x_label': 'X label',
 'y_label': 'Y label'}


In [ ]:
# chart-style：学生修改入口；视觉常量应保留在这个四层映射中。
# chart-style: student editing entry; keep visual constants in this four-layer mapping.
CHART_STYLE = {
    "canvas": {
        "figure_size": [3.35, 2.30], "figure_facecolor": "white", "axes_facecolor": "white",
        "grid": {"show": True, "axis": "y", "color": "#9A9A9A", "linestyle": "--", "linewidth": 0.5, "alpha": 0.5},
        "spines": {"visible": ["left", "right", "top", "bottom"], "color": "#8C8C8C", "linewidth": 0.65, "alpha": 0.80},
    },
    "marks": {
        "colors": ['#0C84C6', '#41B7AC', '#FFA510', '#F74D4D', '#2455A4', '#002C53'],
        "hatches": ["", "//", "\\", "xx", "..", "++"],
        "group_width": 0.80, "edgecolor": "white", "edgewidth": 0.65,
    },
    "legend": {"show": True, "loc": "best", "bbox_to_anchor": None, "ncol": 1, "fontsize": 8.5, "title": None, "title_fontsize": 8.5, "facecolor": "white", "edgecolor": "#A6A6A6", "framealpha": 0.96, "linewidth": 0.65},
    "axes": {
        "label": {"fontsize": 10.0, "fontweight": "bold", "color": "#222222"},
        "x": {"tick_fontsize": 8.5, "tick_fontweight": "bold", "tick_rotation": 0},
        "y": {"scale": "linear", "limits": [0, None], "tick_fontsize": 8.5, "tick_fontweight": "bold", "tick_rotation": 0, "tick_format": None},
    },
}


In [ ]:
required_columns = [PLOT_SPEC["x_column"], PLOT_SPEC["y_column"]]
if PLOT_SPEC.get("series_column"): required_columns.append(PLOT_SPEC["series_column"])
missing_columns = [column for column in required_columns if column not in source_data.columns]
if missing_columns: raise KeyError(f"Missing required columns: {missing_columns}")
if source_data[required_columns].isna().any().any(): raise ValueError("Required plotting columns contain missing values; document a policy before plotting")
# Derived data remains in memory. Add and document justified aggregation here.
plot_data = source_data[required_columns].copy()
plot_data.head()


In [ ]:
def _style_axes(ax):
    canvas, axes = CHART_STYLE["canvas"], CHART_STYLE["axes"]
    ax.set_facecolor(canvas["axes_facecolor"])
    grid = canvas["grid"]; ax.grid(grid["show"], axis=grid["axis"], color=grid["color"], linestyle=grid["linestyle"], linewidth=grid["linewidth"], alpha=grid["alpha"])
    spines = canvas["spines"]
    for name, spine in ax.spines.items():
        spine.set_visible(name in spines["visible"]); spine.set_color(spines["color"]); spine.set_linewidth(spines["linewidth"]); spine.set_alpha(spines["alpha"])
    xcfg, ycfg = axes["x"], axes["y"]
    ax.tick_params(axis="x", labelsize=xcfg["tick_fontsize"]); ax.tick_params(axis="y", labelsize=ycfg["tick_fontsize"])
    for label in ax.get_xticklabels(): label.set_fontweight(xcfg["tick_fontweight"]); label.set_rotation(xcfg["tick_rotation"])
    for label in ax.get_yticklabels(): label.set_fontweight(ycfg["tick_fontweight"]); label.set_rotation(ycfg["tick_rotation"])
    ax.set_yscale(ycfg["scale"])
    if ycfg["limits"] is not None: ax.set_ylim(*ycfg["limits"])
    if ycfg["tick_format"] is not None: ax.yaxis.set_major_formatter(StrMethodFormatter(ycfg["tick_format"]))
    label = axes["label"]; ax.set_xlabel(PLOT_SPEC["x_label"], **label); ax.set_ylabel(PLOT_SPEC["y_label"], **label)

def _add_legend(ax):
    cfg = CHART_STYLE["legend"]
    if not cfg["show"]: return None
    kwargs = {"loc": cfg["loc"], "ncol": cfg["ncol"], "fontsize": cfg["fontsize"], "title": cfg["title"], "title_fontsize": cfg["title_fontsize"], "frameon": True}
    if cfg["bbox_to_anchor"] is not None: kwargs["bbox_to_anchor"] = cfg["bbox_to_anchor"]
    legend = ax.legend(**kwargs); frame = legend.get_frame(); frame.set_facecolor(cfg["facecolor"]); frame.set_edgecolor(cfg["edgecolor"]); frame.set_alpha(cfg["framealpha"]); frame.set_linewidth(cfg["linewidth"]); return legend

def build_figure():
    with plt.rc_context(PAPER_STYLE):
        canvas, marks = CHART_STYLE["canvas"], CHART_STYLE["marks"]
        fig, ax = plt.subplots(figsize=canvas["figure_size"]); fig.patch.set_facecolor(canvas["figure_facecolor"])
        xcol, ycol, series_col = PLOT_SPEC["x_column"], PLOT_SPEC["y_column"], PLOT_SPEC.get("series_column")
        if series_col:
            if plot_data.duplicated([xcol, series_col]).any(): raise ValueError("Grouped bars require one value per category and series; aggregate explicitly first")
            categories = list(dict.fromkeys(plot_data[xcol])); series_order = list(dict.fromkeys(plot_data[series_col]))
            pivot = plot_data.pivot(index=xcol, columns=series_col, values=ycol).reindex(index=categories, columns=series_order)
            positions = np.arange(len(categories)); width = marks["group_width"] / len(series_order)
            for index, series in enumerate(series_order):
                offset = (index - (len(series_order) - 1) / 2) * width
                ax.bar(positions + offset, pivot[series], width, label=str(series), color=marks["colors"][index % len(marks["colors"])], hatch=marks["hatches"][index % len(marks["hatches"])], edgecolor=marks["edgecolor"], linewidth=marks["edgewidth"])
            ax.set_xticks(positions, [str(item) for item in categories]); _add_legend(ax)
        else:
            if plot_data.duplicated([xcol]).any(): raise ValueError("Bars require one value per category; aggregate explicitly first")
            positions = np.arange(len(plot_data)); colors = [marks["colors"][i % len(marks["colors"])] for i in range(len(plot_data))]
            hatches = [marks["hatches"][i % len(marks["hatches"])] for i in range(len(plot_data))]
            ax.bar(positions, plot_data[ycol], marks["group_width"], color=colors, hatch=hatches, edgecolor=marks["edgecolor"], linewidth=marks["edgewidth"]); ax.set_xticks(positions, plot_data[xcol].astype(str))
        _style_axes(ax); return fig


In [ ]:
def atomic_save_figure(figure, destination, dpi=300):
    destination = Path(destination)
    descriptor, temporary_name = tempfile.mkstemp(prefix=f".{destination.stem}.", suffix=destination.suffix, dir=destination.parent); os.close(descriptor)
    try:
        options = {"format": destination.suffix.lstrip("."), "bbox_inches": None, "facecolor": CHART_STYLE["canvas"]["figure_facecolor"]}
        if destination.suffix.lower() == ".png": options["dpi"] = dpi
        figure.savefig(temporary_name, **options); os.replace(temporary_name, destination)
    except Exception:
        Path(temporary_name).unlink(missing_ok=True); raise

FIGURE_METADATA["dimensions_inches"] = list(CHART_STYLE["canvas"]["figure_size"])
FIGURE_METADATA["palette"]["colors"] = list(CHART_STYLE["marks"]["colors"])
FIGURE_METADATA["palette"]["base_style"]["embedded_sha256"] = hashlib.sha256(BASE_MPLSTYLE.encode("utf-8")).hexdigest()
figure = build_figure()
for output_name in FIGURE_METADATA["outputs"]: atomic_save_figure(figure, FIGURE_DIR / output_name)
manifest = dict(FIGURE_METADATA); manifest["environment"] = {"python": platform.python_version(), "matplotlib": matplotlib.__version__, "pandas": pd.__version__}
manifest_path = FIGURE_DIR / f"{FIGURE_METADATA['figure_slug']}.manifest.json"
descriptor, temporary_name = tempfile.mkstemp(prefix=f".{manifest_path.stem}.", suffix=manifest_path.suffix, dir=manifest_path.parent); os.close(descriptor)
try:
    Path(temporary_name).write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"); os.replace(temporary_name, manifest_path)
except Exception:
    Path(temporary_name).unlink(missing_ok=True); raise
figure
